# DeepONet / MNO Benchmark

Simple notebook for one selected model:

```text
[coefficients, initial condition] -> solution trajectory
```

Set `MODEL_NAME = 'deeponet'` for concatenated DeepONet or `MODEL_NAME = 'mno'` for the leaf/branch/trunk MNO variant.


In [ ]:
from pathlib import Path
from time import perf_counter
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader

# Make sure the notebook uses the local files in this folder, and reload them after edits.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'data.py').exists():
    NOTEBOOK_DIR = Path.cwd() / 'neural_network_experiments'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import data as pde_data
import models as pde_models
importlib.reload(pde_data)
importlib.reload(pde_models)

PDESampleDataset = pde_data.PDESampleDataset
fit_normalizers = pde_data.fit_normalizers
load_sample_data = pde_data.load_sample_data
normalize_data = pde_data.normalize_data
TensorizedConcatDeepONet = pde_models.TensorizedConcatDeepONet
MNO = pde_models.MNO

pd.set_option('display.max_columns', 120)


In [ ]:
# Data configuration
FRAMEWORK = 'Framework2'          # 'Framework1' also works with sample-mode logic
PDE_NAME = 'Conservation_law'     # Framework1 wave folder is Parametric_Wave; Framework2 wave folder is Param_wave
TRAIN_SIZE = 1000                 # keep small while debugging
USE_OOD = False

# Optional preprocessing, ported from the older prepare_batch idea
X_NUM_MODEL = None                # None keeps all spatial points; e.g. 64 subsamples space
OUTPUT_START_INDEX = None         # None starts outputs at the input time index
OUTPUT_STEP = 1                   # e.g. 2 predicts every other output time
NORMALIZATION = 'global'          # 'global', 'per_sample_input', or 'none'

# Model configuration
MODEL_NAME = 'DeepONet'           # 'DeepONet' or 'MNO'
HIDDEN_DIM = 200
TRUNK_DEPTH = 2
BRANCH_DEPTH = 2

# DeepONet tensor sizes: branch output size is DEEPONET_NUM_TRUNK * DEEPONET_NUM_BRANCH.
DEEPONET_NUM_TRUNK = 100
DEEPONET_NUM_BRANCH = 100

# MNO tensor sizes: branch output size is MNO_NUM_LEAF * MNO_NUM_TRUNK * MNO_NUM_BRANCH.
# Balanced MNO near DeepONet size: 4 * 50 * 50 = 10,000, matching 100 * 100.
# Original large MNO was 100 * 100 * 100 = 1,000,000.
MNO_NUM_LEAF = 4
MNO_NUM_TRUNK = 50
MNO_NUM_BRANCH = 50
LEAF_DEPTH = 2

# Training configuration
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
MAX_EPOCHS = 10
STEPS_PER_EPOCH = None            # None means one full pass through the training loader per epoch
LR = 1e-4
WEIGHT_DECAY = 1e-4
CLIP_GRAD_NORM = 1.0
USE_AMP = torch.cuda.is_available()
LOG_EVERY = 20
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)
print('device:', DEVICE)

In [ ]:
raw = load_sample_data(
    framework=FRAMEWORK,
    pde=PDE_NAME,
    train_size=TRAIN_SIZE,
    x_num_model=X_NUM_MODEL,
    output_start_index=OUTPUT_START_INDEX,
    output_step=OUTPUT_STEP,
    use_ood=USE_OOD,
)
normalizers = fit_normalizers(raw)
data = normalize_data(raw, normalizers, mode=NORMALIZATION)

train_dataset = PDESampleDataset(data.x_train, data.coef_train, data.y_train, data.y_train_mean, data.y_train_std)
test_dataset = PDESampleDataset(data.x_test, data.coef_test, data.y_test, data.y_test_mean, data.y_test_std)
ood_dataset = (
    PDESampleDataset(data.x_ood, data.coef_ood, data.y_ood, data.y_ood_mean, data.y_ood_std)
    if data.x_ood is not None else None
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)
ood_loader = DataLoader(ood_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False) if ood_dataset is not None else None

n_x = data.x_train.shape[1]
coef_dim = data.coef_train.shape[1]
n_t = data.y_train.shape[1]

print('train:', data.x_train.shape, data.coef_train.shape, data.y_train.shape)
print('test: ', data.x_test.shape, data.coef_test.shape, data.y_test.shape)
print('n_t, n_x, coef_dim:', n_t, n_x, coef_dim)
print('normalization:', NORMALIZATION, '| x_num_model:', X_NUM_MODEL, '| output_step:', OUTPUT_STEP)


In [ ]:
def count_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def make_model(model_name, *, num_leaf=None, num_trunk=None, num_branch=None):
    model_key = model_name.lower()
    if model_key == 'deeponet':
        trunk_width = DEEPONET_NUM_TRUNK if num_trunk is None else num_trunk
        branch_width = DEEPONET_NUM_BRANCH if num_branch is None else num_branch
        return TensorizedConcatDeepONet(
            n_x=n_x,
            coef_dim=coef_dim,
            n_t=n_t,
            num_trunk=trunk_width,
            num_branch=branch_width,
            hidden_dim=HIDDEN_DIM,
            trunk_depth=TRUNK_DEPTH,
            branch_depth=BRANCH_DEPTH,
        )
    if model_key == 'mno':
        leaf_width = MNO_NUM_LEAF if num_leaf is None else num_leaf
        trunk_width = MNO_NUM_TRUNK if num_trunk is None else num_trunk
        branch_width = MNO_NUM_BRANCH if num_branch is None else num_branch
        return MNO(
            n_x=n_x,
            coef_dim=coef_dim,
            n_t=n_t,
            num_leaf=leaf_width,
            num_trunk=trunk_width,
            num_branch=branch_width,
            hidden_dim=HIDDEN_DIM,
            trunk_depth=TRUNK_DEPTH,
            branch_depth=BRANCH_DEPTH,
            leaf_depth=LEAF_DEPTH,
        )
    raise ValueError(f'Unknown MODEL_NAME: {model_name}')


model = make_model(MODEL_NAME).to(DEVICE)
model_key = MODEL_NAME.lower()

n_params = count_parameters(model)
comparison_rows = [
    {
        'model': 'DeepONet selected',
        'num_leaf': np.nan,
        'num_trunk': DEEPONET_NUM_TRUNK,
        'num_branch': DEEPONET_NUM_BRANCH,
        'branch_output_dim': DEEPONET_NUM_TRUNK * DEEPONET_NUM_BRANCH,
        'parameters': count_parameters(make_model('DeepONet')),
    },
    {
        'model': 'MNO selected',
        'num_leaf': MNO_NUM_LEAF,
        'num_trunk': MNO_NUM_TRUNK,
        'num_branch': MNO_NUM_BRANCH,
        'branch_output_dim': MNO_NUM_LEAF * MNO_NUM_TRUNK * MNO_NUM_BRANCH,
        'parameters': count_parameters(make_model('MNO')),
    },
    {
        'model': 'MNO original config',
        'num_leaf': 100,
        'num_trunk': 100,
        'num_branch': 100,
        'branch_output_dim': 100 * 100 * 100,
        'parameters': count_parameters(make_model('MNO', num_leaf=100, num_trunk=100, num_branch=100)),
    },
]
display(pd.DataFrame(comparison_rows))
print(f'Selected model: {MODEL_NAME} | parameters: {n_params:,}')
model

In [ ]:
def relative_l2(pred, target):
    err = torch.sqrt(((target - pred) ** 2).flatten(1).sum(1))
    scale = 1e-7 + torch.sqrt((target ** 2).flatten(1).sum(1))
    return err / scale


def denormalize_batch(y_norm, batch):
    if 'y_mean' in batch and 'y_std' in batch:
        y_mean = batch['y_mean'].to(DEVICE)
        y_std = batch['y_std'].to(DEVICE)
        return y_norm * y_std + y_mean
    if NORMALIZATION == 'none':
        return y_norm
    return normalizers.denormalize_y_torch(y_norm)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_l2 = []
    first_pred = None
    first_target = None
    for batch in loader:
        x = batch['x'].to(DEVICE)
        coeff = batch['coef'].to(DEVICE)
        y_norm = batch['y'].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=torch.bfloat16):
            pred_norm = model(x, coeff)
        pred = denormalize_batch(pred_norm, batch)
        target = denormalize_batch(y_norm, batch)
        all_l2.extend(relative_l2(pred, target).detach().cpu().tolist())
        if first_pred is None:
            first_pred = pred[0].detach().cpu().numpy()
            first_target = target[0].detach().cpu().numpy()
    return float(np.mean(all_l2)), first_pred, first_target


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-6)
steps_per_epoch = STEPS_PER_EPOCH or len(train_loader)
total_steps = MAX_EPOCHS * steps_per_epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

print(f'Training: {MAX_EPOCHS} epochs x {steps_per_epoch} steps = {total_steps} total steps')


In [ ]:
train_losses = []
eval_l2_errors = []
best_l2 = float('inf')
best_state = None
best_prediction = None
best_target = None

start_train = perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    data_iter = iter(train_loader)

    for step in range(steps_per_epoch):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        x = batch['x'].to(DEVICE)
        coeff = batch['coef'].to(DEVICE)
        y = batch['y'].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=torch.bfloat16):
            output = model(x, coeff)
            loss = F.mse_loss(output, y)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
            optimizer.step()

        scheduler.step()
        epoch_loss += float(loss.detach().cpu())

        if (step + 1) % LOG_EVERY == 0 or (step + 1) == steps_per_epoch:
            print(f'  Epoch {epoch:03d} | Step {step + 1:04d}/{steps_per_epoch} | Loss {epoch_loss / (step + 1):.6e}')

    avg_loss = epoch_loss / steps_per_epoch
    train_losses.append(avg_loss)

    mean_l2, first_pred, first_target = evaluate(test_loader)
    eval_l2_errors.append(mean_l2)

    if mean_l2 < best_l2:
        best_l2 = mean_l2
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_prediction = first_pred
        best_target = first_target
        print(f'  >> New best relative L2: {best_l2:.6f}')

    print(f'Epoch {epoch:03d} | Train MSE {avg_loss:.6e} | Test Rel L2 {mean_l2:.6f}')
    print()

train_time = perf_counter() - start_train
print(f'Total train time: {train_time:.2f}s')


In [ ]:
# Restore best model and compute final metrics
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

selected_num_trunk = DEEPONET_NUM_TRUNK if MODEL_NAME.lower() == 'deeponet' else MNO_NUM_TRUNK
selected_num_branch = DEEPONET_NUM_BRANCH if MODEL_NAME.lower() == 'deeponet' else MNO_NUM_BRANCH
selected_num_leaf = MNO_NUM_LEAF if MODEL_NAME.lower() == 'mno' else None

test_l2, test_pred, test_target = evaluate(test_loader)
rows = [{
    'framework': FRAMEWORK,
    'pde': PDE_NAME,
    'model': MODEL_NAME,
    'split': 'test',
    'mean_relative_error': test_l2,
    'n_params': n_params,
    'num_trunk': selected_num_trunk,
    'num_branch': selected_num_branch,
    'num_leaf': selected_num_leaf,
    'n_train': len(train_dataset),
    'n_eval': len(test_dataset),
    'train_time_seconds': train_time,
    'epochs': MAX_EPOCHS,
    'steps_per_epoch': steps_per_epoch,
    'lr': LR,
    'batch_size': BATCH_SIZE,
    'normalization': NORMALIZATION,
    'x_num_model': X_NUM_MODEL,
    'output_step': OUTPUT_STEP,
}]

if ood_loader is not None:
    ood_l2, _, _ = evaluate(ood_loader)
    rows.append({**rows[0], 'split': 'ood', 'mean_relative_error': ood_l2, 'n_eval': len(ood_dataset)})

results_df = pd.DataFrame(rows)
display(results_df)

output_path = Path('neural_network_experiments') / f'results_{FRAMEWORK}_{PDE_NAME}_{MODEL_NAME}.csv'
results_df.to_csv(output_path, index=False)
print('saved', output_path)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(np.arange(1, len(train_losses) + 1), train_losses)
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('train MSE')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(np.arange(1, len(eval_l2_errors) + 1), eval_l2_errors)
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('test relative L2')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize one test trajectory
prediction = test_pred
truth = test_target
abs_error = np.abs(prediction - truth)
rel = np.linalg.norm(prediction - truth) / np.linalg.norm(truth)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), constrained_layout=True)
for ax, title, arr in zip(axes, ['true', 'prediction', f'abs error | rel={rel:.3e}'], [truth, prediction, abs_error]):
    im = ax.imshow(arr, aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
# Optional: save best checkpoint
checkpoint_path = Path('neural_network_experiments') / f'best_{FRAMEWORK}_{PDE_NAME}_{MODEL_NAME}.pt'
torch.save({
    'model_state_dict': best_state,
    'config': {
        'framework': FRAMEWORK,
        'pde': PDE_NAME,
        'model': MODEL_NAME,
        'n_x': n_x,
        'n_t': n_t,
        'coef_dim': coef_dim,
        'num_trunk': selected_num_trunk,
        'num_branch': selected_num_branch,
        'num_leaf': selected_num_leaf,
        'leaf_depth': LEAF_DEPTH if MODEL_NAME.lower() == 'mno' else None,
        'hidden_dim': HIDDEN_DIM,
        'trunk_depth': TRUNK_DEPTH,
        'branch_depth': BRANCH_DEPTH,
        'normalization': NORMALIZATION,
        'x_num_model': X_NUM_MODEL,
        'output_step': OUTPUT_STEP,
    },
    'train_losses': train_losses,
    'eval_l2_errors': eval_l2_errors,
}, checkpoint_path)
print('saved', checkpoint_path)